# 03 — Random Search

Za razliku od Grid Search-a, ovde ne isprobavamo baš sve kombinacije, već nasumično uzorkujemo vrednosti hiperparametara iz zadatih raspodela. Broj iteracija namerno postavljamo na isti broj kao broj kombinacija koje je Grid Search isprobao za taj model, da bi poređenje bilo fer.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
%matplotlib inline

from sklearn.datasets import load_breast_cancer, load_digits
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, KFold, train_test_split
from scipy.stats import loguniform, randint
import time

In [2]:
breast_cancer = load_breast_cancer()
digits = load_digits()

X_bc, y_bc = breast_cancer.data, breast_cancer.target
X_dg, y_dg = digits.data, digits.target

X_bc_train, X_bc_test, y_bc_train, y_bc_test = train_test_split(
    X_bc, y_bc, test_size=0.2, random_state=42, stratify=y_bc)
X_dg_train, X_dg_test, y_dg_train, y_dg_test = train_test_split(
    X_dg, y_dg, test_size=0.2, random_state=42, stratify=y_dg)

skupovi_podataka = {
    'breast_cancer': (X_bc_train, y_bc_train),
    'digits': (X_dg_train, y_dg_train),
}

SEEDOVI = list(range(10))
BROJ_FOLDOVA = 3  

## Raspodele za nasumično uzorkovanje

Koristimo `loguniform` za `C` i `gamma` kod SVM-a jer te vrednosti obično imaju smisla na logaritamskoj skali. Kod Random Forest-a koristimo `randint` jer su ti hiperparametri celi brojevi.

In [3]:
param_dist_svm = {
    'C': loguniform(1e-2, 1e2),
    'gamma': loguniform(1e-4, 1e0),
}

param_dist_rf = {
    'n_estimators': randint(50, 300),
    'max_depth': randint(2, 20),
    'min_samples_split': randint(2, 10),
}

n_iter_svm = 25
n_iter_rf = 60

## Pokretanje 

In [4]:
rezultati_random = []

for ime_skupa, (X, y) in skupovi_podataka.items():
    for ime_modela in ['svm', 'random_forest']:

        skorovi_po_seedu = []
        vremena_po_seedu = []
        parametri_po_seedu = []

        if ime_modela == 'svm':
            raspodela = param_dist_svm
            n_iter = n_iter_svm
        else:
            raspodela = param_dist_rf
            n_iter = n_iter_rf

        for seed in SEEDOVI:
            kf = KFold(n_splits=BROJ_FOLDOVA, shuffle=True, random_state=seed)

            if ime_modela == 'svm':
                model = SVC(kernel='rbf', random_state=42)
            else:
                model = RandomForestClassifier(random_state=seed, n_jobs=-1)

            pocetak = time.time()
            random_search = RandomizedSearchCV(
                model, raspodela, n_iter=n_iter, cv=kf, scoring='accuracy',
                random_state=seed, n_jobs=-1,
            )
            random_search.fit(X, y)
            trajanje = time.time() - pocetak

            skorovi_po_seedu.append(random_search.best_score_)
            vremena_po_seedu.append(trajanje)
            parametri_po_seedu.append(random_search.best_params_)

        indeks_najboljeg = int(np.argmax(skorovi_po_seedu))

        rezultati_random.append({
            'dataset': ime_skupa,
            'model': ime_modela,
            'method': 'random_search',
            'mean_score': np.mean(skorovi_po_seedu),
            'std_score': np.std(skorovi_po_seedu),
            'n_evaluations': n_iter,
            'mean_time_sec': np.mean(vremena_po_seedu),
            'best_params': parametri_po_seedu[indeks_najboljeg],
        })

        print(f'{ime_skupa:15s} {ime_modela:15s} '
              f'mean_score={np.mean(skorovi_po_seedu):.4f} (+/- {np.std(skorovi_po_seedu):.4f})  '
              f'mean_time={np.mean(vremena_po_seedu):6.1f}s')

breast_cancer   svm             mean_score=0.9391 (+/- 0.0052)  mean_time=   0.2s
breast_cancer   random_forest   mean_score=0.9629 (+/- 0.0029)  mean_time=  11.9s
digits          svm             mean_score=0.9892 (+/- 0.0020)  mean_time=   1.7s
digits          random_forest   mean_score=0.9745 (+/- 0.0017)  mean_time=  17.2s


## Rezultati

In [5]:
df_random = pd.DataFrame(rezultati_random)
df_random

,dataset,model,method,mean_score,std_score,n_evaluations,mean_time_sec,best_params
0,breast_cancer,svm,random_search,0.939116,0.005188,25,0.234502,"{'C': 0.5545996722241042, 'gamma': 0.000126971..."
1,breast_cancer,random_forest,random_search,0.962851,0.002863,60,11.869927,"{'max_depth': 17, 'min_samples_split': 2, 'n_e..."
2,digits,svm,random_search,0.989214,0.002047,25,1.675994,"{'C': 24.335817588808077, 'gamma': 0.000208248..."
3,digits,random_forest,random_search,0.974530,0.001710,60,17.155604,"{'max_depth': 16, 'min_samples_split': 2, 'n_e..."


In [6]:
kolone_rezultata = ['dataset', 'model', 'method', 'mean_score', 'std_score',
                    'n_evaluations', 'mean_time_sec', 'best_params']

import os
os.makedirs('./results', exist_ok=True)

df_random[kolone_rezultata].to_csv('./results/all_results.csv', mode='a', header=False, index=False)
print('Rezultati sačuvani u results/all_results.csv')

Rezultati sačuvani u results/all_results.csv
